### Jupyter Notebook to test code in cells before adding to Python script in .py file

### Data sources:
- populations https://www.michigan-demographics.com/counties_by_population
- geopoints https://public.opendatasoft.com/explore/dataset/us-county-boundaries/export/?flg=en-us&disjunctive.statefp&disjunctive.countyfp&disjunctive.name&disjunctive.namelsad&disjunctive.stusab&disjunctive.state_name&sort=stusab
- shapefile https://public.opendatasoft.com/explore/dataset/us-county-boundaries/export/?flg=en-us&disjunctive.statefp&disjunctive.countyfp&disjunctive.name&disjunctive.namelsad&disjunctive.stusab&disjunctive.state_name&sort=stusab&refine.statefp=26

In [1]:
# Import necessary libraries
import numpy as np
import geopandas as gpd   
import pandas as pd 
from math import pi, pow, sin, cos, asin, sqrt, floor
from pulp import *

#### Precursor 1: Read in the Excel file and GeoJSON file


In [2]:
# Read in the Excel file of Michigan counties
xlsx_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties.xlsx'
michigan_counties = pd.read_excel(xlsx_file_path)

# Read in the GeoJSON file of Michigan county boundaries
geojson_file_path = '/Users/stefanjenss/Desktop/DataScience/Decision_analytics/Module6/michigan_counties.geojson'
michigan_counties_geojson = gpd.read_file(geojson_file_path)

# michigan_counties.head()
# michigan_counties_geojson.head()

#### Precursor 2: Merge the two data sources on country names to have a comprehensive dataset.
- In the 'michigan_counties' dataframe, the county name column is 'county_names', and in the 'michigan_counties_geojson' datafram, the county name column is 'name'.

In [3]:
# Merge the two dataframes on the county name
michigan_counties_merged = michigan_counties.merge(michigan_counties_geojson, left_on='county_names', right_on='name', how='inner')

# michigan_counties_merged.head()

#### Precursor 2: Obtain adjacency information for the countries:
- This will be done by checking which counties share a coundary using the GeoJSON file.

In [4]:
# Determine adjacency between counties based on their geometries
adjacency_matrix = []

# Loop through each county in the GeoJSON file and determine its adjacency to all other counties
for index_1, county_1 in michigan_counties_merged.iterrows(): # Loop through each county in the GeoJSON file
    neighbors = [] # Create an empty list to store the names of neighboring counties
    for index_2, county_2 in michigan_counties_merged.iterrows(): # Loop through each county in the GeoJSON file
        # Check if the two counties are the same to avoid self-comparison
        if county_1['name'] != county_2['name']:
            if county_1['geometry'].touches(county_2['geometry']): # Check if the two counties touch
                neighbors.append(county_2['name']) # If they do, add the name of the second county to the list of neighbors
    adjacency_matrix.append(neighbors) # Add the list of neighbors to the adjacency matrix

# Add adjacency information to the dataframe
michigan_counties_merged['adjacent_counties'] = adjacency_matrix

# Check the adjacency information for the first few counties
michigan_counties_merged[['name', 'adjacent_counties']].head()

,name,adjacent_counties
0,Leelanau,"[Schoolcraft, Antrim, Benzie, Grand Traverse, ..."
1,Clinton,"[Ionia, Shiawassee, Ingham, Montcalm, Eaton, G..."
2,Wexford,"[Missaukee, Osceola, Manistee, Benzie, Lake, K..."
3,Branch,"[Calhoun, St. Joseph, Kalamazoo, Hillsdale]"
4,Ionia,"[Clinton, Barry, Kent, Montcalm, Eaton, Gratiot]"


### Section 1: Calculation Distance Between County Pairs

In [5]:
# Define function to calculate distance between two sets of longitudes / latitudes

# Function to convert degrees to radians
def degrees_to_radians(x):
    return((pi / 180) * x)

# Function to calculate distance between two points on a sphere (in miles) given their longitudes and latitudes
def lon_lat_distance_miles(lon_a, lat_a, lon_b, lat_b):
    """
    Calculates the great-circle distance between two points on a sphere given their longitudes and latitudes.
    
    Parameters:
    lon_a (float): longitude of point A in degrees
    lat_a (float): latitude of point A in degrees
    lon_b (float): longitude of point B in degrees
    lat_b (float): latitude of point B in degrees
    
    Returns:
    float: distance between the two points in miles
    """
    radius_of_earth = 24872 / (2 * pi)
    c = sin((degrees_to_radians(lat_a) - \
    degrees_to_radians(lat_b)) / 2)**2 + \
    cos(degrees_to_radians(lat_a)) * \
    cos(degrees_to_radians(lat_b)) * \
    sin((degrees_to_radians(lon_a) - \
    degrees_to_radians(lon_b))/2)**2
    return(2 * radius_of_earth * (asin(sqrt(c))))

# Function to convert the distance between two points on a sphere (in miles) to meters
def lon_lat_distance_meters (lon_a, lat_a, lon_b, lat_b):
    return(lon_lat_distance_miles(lon_a, lat_a, lon_b, lat_b) * 1609.34)

In [6]:
# Remove population to allow easy joining of long and lat for each county pair
lat_lon = ['county_names', 'latitude', 'longitude']
lat_lon = michigan_counties_merged[lat_lon]

# Create list of county names for pairing        
county_names = michigan_counties['county_names'].to_numpy()

# Create each unique pair
pairs = []

# Loop through each county name and create a pair with each other county name
for i in range(len(county_names)):
    for j in range(i + 1, len(county_names)):
        pairs.append((county_names[i], county_names[j]))

# Create column names for county pairs df
col_names = ['county_1', 'county_2']

# Create df of county pairs                
county_pairs = pd.DataFrame(pairs, columns = col_names)

 # Add first county longitude and latitude
county_pairs = county_pairs.merge(lat_lon, left_on = 'county_1', right_on = 'county_names', how = 'left')
county_pairs.drop('county_names', axis = 1, inplace = True) # Drop county names column
county_pairs = county_pairs.rename(columns={'latitude': 'county_1_lat', 'longitude': 'county_1_long'}) # Rename columns

# Add second county longitude and latitude
county_pairs = county_pairs.merge(lat_lon, left_on = 'county_2', right_on = 'county_names', how = 'left')
county_pairs.drop('county_names', axis = 1, inplace = True) # Drop county names column
county_pairs = county_pairs.rename(columns={'latitude': 'county_2_lat', 'longitude': 'county_2_long'}) # Rename columns

# Add distance between each county pair in miles and meters;
distance_miles = [] # Create empty list to store distance in miles
distance_meters = [] # Create empty list to store distance in meters

# Loop through each county pair and calculate distance in miles and meters
for i in range(len(county_pairs)):
    distance_miles.append(lon_lat_distance_miles(county_pairs.iloc[i, 2], county_pairs.iloc[i, 3], county_pairs.iloc[i, 4], county_pairs.iloc[i, 5]))
    distance_meters.append(lon_lat_distance_meters(county_pairs.iloc[i, 2], county_pairs.iloc[i, 3], county_pairs.iloc[i, 4], county_pairs.iloc[i, 5]))

# Add distance columns to county pairs df
county_pairs['distance_miles'] = distance_miles
county_pairs['distance_meters'] = distance_meters

# Check table
county_pairs

,county_1,county_2,county_1_lat,county_1_long,county_2_lat,county_2_long,distance_miles,distance_meters
0,Leelanau,Clinton,45.151771,-86.038496,42.943652,-84.601517,100.038253,160995.562830
1,Leelanau,Wexford,45.151771,-86.038496,44.338367,-85.578414,32.050066,51579.452482
2,Leelanau,Branch,45.151771,-86.038496,41.916119,-85.059044,69.831366,112382.410744
3,Leelanau,Ionia,45.151771,-86.038496,42.945094,-85.074603,67.621439,108825.886417
4,Leelanau,Mecosta,45.151771,-86.038496,43.640768,-85.324634,49.938224,80367.581755
...,...,...,...,...,...,...,...,...
3398,Arenac,Alcona,44.042885,-83.747242,44.683623,-83.129008,43.010988,69219.303841
3399,Arenac,Livingston,44.042885,-83.747242,42.602917,-83.911528,15.593543,25095.312007
3400,Charlevoix,Alcona,45.502498,-85.373250,44.683623,-83.129008,155.151831,249692.047763
3401,Charlevoix,Livingston,45.502498,-85.373250,42.602917,-83.911528,102.674475,165238.138922


In [8]:
### Jaimon's Code ###

### Objective function and contraints ###

# Assume n_counties and n_districts are predefined
n_counties = 83
n_districts = 14

# Declare the problem to be a minimization problem
model = LpProblem('Compacted-Redisticting', LpMinimize)


## Declare the decision variables
# Decision variable: binary variable for county to district assignment
x = [[LpVariable(f'x_{i}_{j}', cat='Binary') for j in range(n_districts)] for i in range(n_counties)]

# Decision variable: Integer variable for population allocation from county to district
population_allocation = [[LpVariable(f'pop_{i}_{j}', lowBound=0, cat='Integer') for j in range(n_districts)] for i in range(n_counties)]


## Define the objective function
# Objective: Minimize the number of counties that are assigned to muliple districts
model += lpSum([x[i][j] for i in range(n_counties) for j in range(n_districts)]) - n_counties

## Define the constraints
# Constraint: Each county can only be assigned to exactly one district
for i in range(n_counties):
    model += lpSum([x[i][j] for j in range(n_districts)]) == 1

# Constraint: Allocate 100% of the population of each county to a district
for i in range(n_counties):
    model += lpSum([population_allocation[i][j] for j in range(n_districts)]) == county_population[i], f"Allocate_All_{i}"

# Ensure that the population allocation is consistent with the county to district assignment
for i in range(n_counties):
    for j in range(n_districts):
        model += population_allocation[i][j] <= county_population[i] * x[i][j]

# Respect for Existing Boundaries: It's often preferable to align districts with existing political boundaries, like county or city limits.
for i in range(n_counties):
    for j in range(n_districts):
        model += x[i][j] <= 1  # If a county is split, it should be split into no more than 2 districts.


# Now, you can solve the model (assuming you have a solver like CBC installed)
model.solve()

# And then, you can retrieve the results or the status of the optimization.
print("Status:", LpStatus[model.status])


# Print the results
for j in range(n_districts):
    district_counties = [i for i in range(n_counties) if x[i][j].varValue == 1]
    print(f"District {j+1} contains counties: {district_counties}")




NameError: name 'county_population' is not defined